<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/MapReduce_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모듈을 설치한다.

In [ ]:
pip install -q langchain_openai langchain_text_splitters langchain_core langchain_core

In [14]:
from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

mobydick 변수에 소설의 일부를 담아 둔다. <BR> 인터넷의 거대 데이터라고 생각하면 된다.

In [15]:
mobydick = """
  CHAPTER 1. Loomings.
Call me Ishmael. Some years ago—never mind how long precisely—having
little or no money in my purse, and nothing particular to interest me
on shore, I thought I would sail about a little and see the watery part
of the world. It is a way I have of driving off the spleen and
regulating the circulation. Whenever I find myself growing grim about
the mouth; whenever it is a damp, drizzly November in my soul; whenever
I find myself involuntarily pausing before coffin warehouses, and
bringing up the rear of every funeral I meet; and especially whenever
my hypos get such an upper hand of me, that it requires a strong moral
principle to prevent me from deliberately stepping into the street, and
methodically knocking people’s hats off—then, I account it high time to
get to sea as soon as I can. This is my substitute for pistol and ball.
With a philosophical flourish Cato throws himself upon his sword; I
quietly take to the ship. There is nothing surprising in this. If they
but knew it, almost all men in their degree, some time or other,
cherish very nearly the same feelings towards the ocean with me.

There now is your insular city of the Manhattoes, belted round by
wharves as Indian isles by coral reefs—commerce surrounds it with her
surf. Right and left, the streets take you waterward. Its extreme
downtown is the battery, where that noble mole is washed by waves, and
cooled by breezes, which a few hours previous were out of sight of
land. Look at the crowds of water-gazers there.

Circumambulate the city of a dreamy Sabbath afternoon. Go from Corlears
Hook to Coenties Slip, and from thence, by Whitehall, northward. What
do you see?—Posted like silent sentinels all around the town, stand
thousands upon thousands of mortal men fixed in ocean reveries. Some
leaning against the spiles; some seated upon the pier-heads; some
looking over the bulwarks of ships from China; some high aloft in the
rigging, as if striving to get a still better seaward peep. But these
are all landsmen; of week days pent up in lath and plaster—tied to
counters, nailed to benches, clinched to desks. How then is this? Are
the green fields gone? What do they here?
"""

필요한 모듈을 임포트 한다.

In [16]:
from collections import defaultdict
from pprint import pprint
import re

원래 문서(mobydick)를 30자씩 분할 한다.

In [32]:
chunks = TokenTextSplitter(chunk_size=30, chunk_overlap=0).split_text(mobydick)

In [ ]:
pprint(chunks)

이제 MAP 과정을 위해 MAP을 담을 배열을 준비한다.

In [34]:
mapped_results = []

MAP을 시작한다.

In [35]:
for i, text_chunk in enumerate(chunks, start=1):  # 각 chunk 번호(i)와 내용(text_chunk)을 함께 받음
    pairs = []  # 현재 chunk에서 만들어질 (단어, 1) 형태의 중간 결과를 저장할 빈 리스트
    words = re.findall(r"\b\w+\b", text_chunk.lower())  # chunk를 소문자로 바꾼 뒤, 정규표현식으로 단어들만 추출

    for word in words:  # 추출한 각 단어를 하나씩 순회
        pairs.append((word, 1))  # 각 단어를 (단어, 1)의 키-값 쌍 변환 ==> 값을 더하면 키가 나왔던 빈도수가 됨
    mapped_results.append(pairs)  # 현재 chunk에서 만든 모든 (단어, 1) 쌍을 mapped_results에 저장


REDUCE 과정 시작: 결과를 저장할 result 딕셔너리 생성

In [44]:
result = defaultdict(int)

REDUCE: 각 키 별로 value(=1)를 더해 몇 번 나왔는지 합산

In [45]:
for pairs in mapped_results:
    for key, value in pairs:
        result[key] += value

최종 결과를 저장하며, 빈도순으로 Sorting

In [46]:
sorted_result1 = dict(sorted(result.items(), key=lambda x: x[1], reverse=True))

결과를 출력해 본다.

In [ ]:
pprint(sorted_result1)

SPARK를 사용한 사례

In [ ]:
from pyspark.sql import SparkSession
import re

spark = SparkSession.builder.appName("wordcount").getOrCreate()


rdd = spark.sparkContext.parallelize(chunks)

result = (
    rdd
    .flatMap(lambda chunk: re.findall(r"\b\w+\b", chunk.lower()))
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

# 빈도 내림차순, 단어 오름차순으로 정렬
sorted_result2 = sorted(result, key=lambda x: x[0])

for word, count in sorted_result2:
    print(f"{word}: {count}")